# 🛠️ Sentiment Analysis Pipeline ```MultinomialNB```

We'll use our preprocessed data to build the text classification pipeline.

In [1]:
import joblib
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

## Load in the *cleaned* Dataset

In [2]:
train_df = pd.read_csv('data/full_cleaned_dataset.csv', header=0, names=['review','label'])

In [3]:
train_df.head()

,review,label
0,"I recommend the book ""Gurdjieff- A Beginner's ...",1
1,I have read two other books about Gladys Aylwa...,0
2,If they cost less I would give them 5 stars. p...,1
3,Tom Sharpe is possibly the funniest author I h...,1
4,Ice cream is a bit too runny.Cheap plastic con...,0


In [4]:
train_df.shape

(3990852, 2)

## Reduce the Dataset

😱 **4 million rows of data!!**
Don't worry if you don't have the computational resources to process that much data. The goal is to demonstrate the *ML Engineering workflow* so we'll reduce the size.

⚠️ **Remember**: Preserve the class balance otherwise you'll introduce bias into the pipelines.

In [5]:
positives = train_df[train_df['label'] == 1]\
                .sample(frac=1, random_state=1, ignore_index=True)\
                .loc[:5_000, ['review','label']]

negatives = train_df[train_df['label'] == 0]\
                .sample(frac=1, random_state=1, ignore_index=True)\
                .loc[:5_000, ['review','label']]

# Make sure same number of classes
negatives.shape == positives.shape

True

In [6]:
final_train_df = pd.concat([positives, negatives], axis=0, ignore_index=True)\
                   .sample(frac=1, random_state=1, ignore_index=True)
final_train_df.shape

(10002, 2)

## Train/Test Split 

We'll use an 80/20 train/test split. The train set will be used for 5-fold Cross Validation hyperparameter tuning and pipeline development.

In [7]:
X, y = final_train_df['review'], final_train_df['label']
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2, 
                                                    train_size=0.8, 
                                                    stratify=y, 
                                                    random_state=1)

## Build the Pipeline

Our pipeline will consist of the ```CountVectorizer``` tokenizer to transform the text input into numerical vectors and finally the ```MultinomialNB``` which will predict the sentiment given the vectors.

In [8]:
nbayes = Pipeline([('vectorizer', CountVectorizer()), ('clfNB', MultinomialNB())])

## Hyperparameter Tuning (Find the Best Pipeline)

We'll use ```GridSearchCV``` to determine the combination of hyperparameters that produce the pipeline with the highest f1 score.

In [9]:
param_grid = {
              'vectorizer__max_features':[600,700,800],
              'clfNB__alpha': [0.1,0.5,1.0]
             }
grid_search = GridSearchCV(nbayes, param_grid, cv=5, scoring='f1', return_train_score=True)
grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...inomialNB())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'clfNB__alpha': [0.1, 0.5, ...], 'vectorizer__max_features': [600, 700, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t

In [10]:
cvres = grid_search.cv_results_
for mean_score, params in zip(cvres['mean_test_score'], cvres['params']):
    print(mean_score)
    print(params)
    print()

0.7931862135939178
{'clfNB__alpha': 0.1, 'vectorizer__max_features': 600}

0.7950848879490922
{'clfNB__alpha': 0.1, 'vectorizer__max_features': 700}

0.8012493978416961
{'clfNB__alpha': 0.1, 'vectorizer__max_features': 800}

0.7935847223883862
{'clfNB__alpha': 0.5, 'vectorizer__max_features': 600}

0.7953821266322327
{'clfNB__alpha': 0.5, 'vectorizer__max_features': 700}

0.8016446382251065
{'clfNB__alpha': 0.5, 'vectorizer__max_features': 800}

0.7936337536816754
{'clfNB__alpha': 1.0, 'vectorizer__max_features': 600}

0.7955871804384751
{'clfNB__alpha': 1.0, 'vectorizer__max_features': 700}

0.801942604454411
{'clfNB__alpha': 1.0, 'vectorizer__max_features': 800}



In [11]:
print(grid_search.best_estimator_)

Pipeline(steps=[('vectorizer', CountVectorizer(max_features=800)),
                ('clfNB', MultinomialNB())])


⚠️ Save the best pipeline!

In [12]:
best_nbayes = grid_search.best_estimator_

## Train and Evaluate the Best Pipeline

⚠️ DO NOT FORGET! Use the **best** pipeline for training and evaluation.

In [13]:
best_nbayes.fit(X_train, y_train)

print(f'f1_score: {f1_score(y_test, best_nbayes.predict(X_test))}')

f1_score: 0.79296875


## 🎉 Save the Best Pipeline 

We'll use the ```joblib``` module to save our pipeline, so we can use our pipeline to make predictions without having to retrain every time.

In [14]:
joblib.dump(best_nbayes, "pipelines/nbayes_pipeline.joblib")

['pipelines/nbayes_pipeline.joblib']